# Basis

## Scrabble

Opdracht: [Scrabble](/problems/10_basis)

## Stap 1: `letter_score(c)`

Eén tak per rij van de tabel met tegelwaarden. De `else` vangt alles af wat geen
kleine letter is.

In [ ]:
def letter_score(c):
    """Geeft de waarde van de Scrabbletegel met letter c, of 0 als c geen letter is."""
    if c in "adeinorst":
        return 1
    elif c in "ghl":
        return 2
    elif c in "bcmp":
        return 3
    elif c in "jkuvw":
        return 4
    elif c == "f":
        return 5
    elif c == "z":
        return 6
    elif c in "xy":
        return 8
    elif c == "q":
        return 10
    else:
        return 0


assert letter_score("w") == 4
assert letter_score("q") == 10
assert letter_score("e") == 1
assert letter_score("%") == 0

## Stap 2: `scrabble_score(s)`

In [ ]:
def scrabble_score(s):
    """Geeft de Scrabblescore van s."""
    if s == "":
        return 0
    else:
        return letter_score(s[0]) + scrabble_score(s[1:])


assert scrabble_score("quotums") == 24
assert scrabble_score("jacquet") == 24
assert scrabble_score("pyjama") == 20
assert scrabble_score("abcdefghijklmnopqrstuvwxyz") == 84
assert scrabble_score("?!@#$%^&*()") == 0
assert scrabble_score("") == 0

## Stap 3: `double_letter(s)`

De drie assertions direct onder de functie zijn een voorbeeld van wat je zelf kon
schrijven: een woord met `True` als uitkomst, een met `False`, en de lege string.
Daarna volgen de assertions uit de testcel.

In [ ]:
def double_letter(s):
    """Geeft True als er ergens in s twee dezelfde letters naast elkaar staan."""
    if len(s) < 2:
        return False
    elif s[0] == s[1]:
        return True
    else:
        return double_letter(s[1:])


assert double_letter("boom") == True
assert double_letter("kat") == False
assert double_letter("") == False
assert double_letter("pizza") == True
assert double_letter("taart") == True
assert double_letter("quiz") == False
assert double_letter("a") == False
assert double_letter("") == False

## Stap 4: `high_scoring(words, n)`

In [ ]:
def high_scoring(words, n):
    """Geeft de woorden uit words met een Scrabblescore van minstens n."""
    if words == []:
        return []
    elif scrabble_score(words[0]) >= n:
        return [words[0]] + high_scoring(words[1:], n)
    else:
        return high_scoring(words[1:], n)


assert high_scoring(["kat", "quiz", "zee", "pizza"], 10) == ["quiz", "pizza"]
assert high_scoring(["kat", "zee"], 10) == []
assert high_scoring([], 10) == []

## Stap 5: `best_word(words)`

Door `>=` wint bij gelijke score het eerste woord.

In [ ]:
def best_word(words):
    """Geeft het woord uit words met de hoogste Scrabblescore; words is niet leeg."""
    if len(words) == 1:
        return words[0]

    rest = best_word(words[1:])

    if scrabble_score(words[0]) >= scrabble_score(rest):
        return words[0]
    else:
        return rest


assert best_word(["kat", "quiz", "zee", "pizza"]) == "quiz"
assert best_word(["kat", "tak"]) == "kat"
assert best_word(["zee"]) == "zee"

## Stap 6: woorden met je eigen tegels

Eén mogelijke uitwerking. De aanpak bestaat uit drie kleine functies, elk met een list
comprehension. `count_letter` telt met `sum` hoe vaak een letter voorkomt. `fits` bepaalt
welke letters van een woord te vaak nodig zijn: een woord past als die lijst leeg is.
`playable_words` filtert daarna de woorden die passen.

Door dat filteren blijven de volgorde en dubbele woorden bewaard. De voorraad
verandert niet, dus elk woord begint met de hele voorraad.


In [ ]:
def count_letter(letter, s):
    """Geeft hoe vaak letter voorkomt in s."""
    return sum([1 for c in s if c == letter])


assert count_letter("a", "aakt") == 2
assert count_letter("z", "aakt") == 0
assert count_letter("a", "") == 0


def fits(tiles, word):
    """Geeft True als word met de letters uit de voorraad tiles gelegd kan worden."""
    too_many = [
        letter for letter in word
        if count_letter(letter, word) > count_letter(letter, tiles)
    ]
    return too_many == []


assert fits("aakt", "kat") == True
assert fits("aakt", "kaak") == False
assert fits("", "") == True


def playable_words(tiles, words):
    """Geeft de woorden die afzonderlijk met de voorraad tiles gelegd kunnen worden."""
    return [word for word in words if fits(tiles, word)]


assert playable_words("aakt", ["kat", "taak", "tak", "kat", "kaak", ""]) == [
    "kat", "taak", "tak", "kat", ""
]
assert playable_words("a", ["aa", "a"]) == ["a"]
assert playable_words("", ["a", ""]) == [""]
assert playable_words("abc", []) == []


## Opdracht 1: een andere winnaar bij gelijke score

1. `letter_score` kent tegelwaarden toe; `scrabble_score` telt ze voor een woord
   op. `double_letter` zoekt gelijke buren, `high_scoring` filtert op een
   minimumscore en `best_word` kiest één winnaar.
2. `scrabble_score` roept `letter_score` en zichzelf aan. `double_letter` roept
   zichzelf aan. `high_scoring` roept zichzelf en `scrabble_score` aan en gebruikt
   daardoor indirect `letter_score`. `best_word` roept zichzelf en
   `scrabble_score` aan en gebruikt indirect `letter_score`.
3. Alleen de vergelijking in `best_word` verandert van `>=` naar `>`.
   De scoreberekening kent geen winnaars; de selectie hoort bij `best_word`.
   Bij gelijkspel wint nu `rest`, dat recursief al het laatste beste woord bevat.
4. `kat` wordt `tak`; bij `kat`, `tak`, `akt` wint `akt`. Een unieke hoogste
   score en één woord blijven gelijk. `high_scoring`, `double_letter`,
   `scrabble_score` en `letter_score` gebruiken `best_word` niet en veranderen
   niet. De assertion voor `kat`/`tak` moet naar `tak`; de andere bestaande
   assertions van stap 5 blijven geldig.

Hieronder staat alleen voor de controle de gewijzigde selectie.


In [ ]:
def best_word(words):
    """Geeft het laatste woord met de hoogste score; words is niet leeg."""
    if len(words) == 1:
        return words[0]
    rest = best_word(words[1:])
    if scrabble_score(words[0]) > scrabble_score(rest):
        return words[0]
    return rest


assert best_word(["kat", "tak"]) == "tak"
assert best_word(["kat", "tak", "akt"]) == "akt"
assert best_word(["kat", "quiz", "zee", "pizza"]) == "quiz"
assert best_word(["zee"]) == "zee"
